# Capital One Mobile · Data Aggregation for Tableau

**Purpose:** Transform raw categorized reviews into 4 aggregated datasets for Tableau dashboard.

**Input:** `cap1_android_categorized.csv` (output from notebook 01)  
**Outputs:**
- `tab_01_weekly_trend.csv` — weekly KPI metrics with WoW deltas
- `tab_02_pain_points.csv` — complaint category breakdown by week and version
- `tab_03_version_impact.csv` — per-version performance with worst/best/current flags
- `tab_04_voc_feed.csv` — curated sample of representative reviews

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('cap1_android_categorized.csv')
df['at'] = pd.to_datetime(df['at'])
df['week'] = df['at'].dt.to_period('W').apply(lambda x: x.start_time.date())
df['is_negative'] = df['score'] <= 2
df['is_positive'] = df['score'] >= 4

print(f"Loaded: {len(df):,} reviews")
print(f"Date range: {df['at'].min().date()} → {df['at'].max().date()}")

## 2. tab_01 · Weekly Trend

One row per week. Includes WoW deltas for KPI tiles.

In [ ]:
weekly = df.groupby('week').agg(
    review_count=('score', 'count'),
    avg_rating=('score', 'mean'),
    negative=('is_negative', 'sum'),
    positive=('is_positive', 'sum'),
    replied_count=('replyContent', lambda x: x.notna().sum())
).reset_index()

weekly['pct_negative'] = (weekly['negative'] / weekly['review_count'] * 100).round(2)
weekly['pct_positive'] = (weekly['positive'] / weekly['review_count'] * 100).round(2)
weekly['avg_rating'] = weekly['avg_rating'].round(2)

# WoW deltas
weekly['rating_wow'] = weekly['avg_rating'].diff().round(2)
weekly['volume_wow_pct'] = (weekly['review_count'].pct_change() * 100).round(1)
weekly['pct_negative_wow_pp'] = weekly['pct_negative'].diff().round(1)
weekly['pct_positive_wow_pp'] = weekly['pct_positive'].diff().round(1)

tab_01 = weekly[['week', 'review_count', 'avg_rating', 'pct_negative', 'pct_positive',
                  'replied_count', 'rating_wow', 'volume_wow_pct',
                  'pct_negative_wow_pp', 'pct_positive_wow_pp']]

tab_01.to_csv('tab_01_weekly_trend.csv', index=False)
print(f"tab_01: {len(tab_01)} weeks")
tab_01.tail(3)

## 3. tab_02 · Pain Points by Week and Version

Complaint category breakdown. Two views: by_week and by_version.

In [ ]:
neg = df[df['is_negative']].copy()

# By week
by_week = neg.groupby(['week', 'category']).agg(
    complaint_count=('score', 'count')
).reset_index()
week_totals = neg.groupby('week').size().reset_index(name='total_in_slice')
by_week = by_week.merge(week_totals, on='week')
by_week['pct_of_slice'] = (by_week['complaint_count'] / by_week['total_in_slice'] * 100).round(1)
by_week.insert(0, 'view_type', 'by_week')
by_week.rename(columns={'week': 'view_value'}, inplace=True)

# By version
by_version = neg.groupby(['reviewCreatedVersion', 'category']).agg(
    complaint_count=('score', 'count')
).reset_index()
version_totals = neg.groupby('reviewCreatedVersion').size().reset_index(name='total_in_slice')
by_version = by_version.merge(version_totals, on='reviewCreatedVersion')
by_version['pct_of_slice'] = (by_version['complaint_count'] / by_version['total_in_slice'] * 100).round(1)
by_version.insert(0, 'view_type', 'by_version')
by_version.rename(columns={'reviewCreatedVersion': 'view_value'}, inplace=True)

tab_02 = pd.concat([by_week, by_version], ignore_index=True)
tab_02.to_csv('tab_02_pain_points.csv', index=False)
print(f"tab_02: {len(tab_02)} rows")
tab_02.head(3)

## 4. tab_03 · Version Impact

Per-version aggregation with flags: Is Worst, Is Best, Is Current.

In [ ]:
ver = df[df['reviewCreatedVersion'].notna()].copy()

tab_03 = ver.groupby('reviewCreatedVersion').agg(
    review_count=('score', 'count'),
    avg_rating=('score', 'mean'),
    first_seen=('at', 'min'),
    last_seen=('at', 'max'),
    pct_negative=('is_negative', 'mean')
).reset_index()

tab_03.columns = ['Version', 'Review Count', 'Avg Rating', 'First Seen Date', 'Last Seen Date', 'Pct Negative']
tab_03['Avg Rating'] = tab_03['Avg Rating'].round(2)
tab_03['Pct Negative'] = (tab_03['Pct Negative'] * 100).round(1)

# Filter to versions with enough reviews
tab_03 = tab_03[tab_03['Review Count'] >= 50].copy()

# Flags
bottom_5 = tab_03.nsmallest(5, 'Avg Rating')['Version'].tolist()
top_5 = tab_03.nlargest(5, 'Avg Rating')['Version'].tolist()
current = tab_03.loc[tab_03['Last Seen Date'].idxmax(), 'Version']

tab_03['Is Worst'] = tab_03['Version'].isin(bottom_5)
tab_03['Is Best'] = tab_03['Version'].isin(top_5)
tab_03['Is Current'] = tab_03['Version'] == current

tab_03 = tab_03.sort_values('First Seen Date')
tab_03.to_csv('tab_03_version_impact.csv', index=False)
print(f"tab_03: {len(tab_03)} versions")
print(f"Current: {current}")
tab_03[tab_03['Is Worst'] | tab_03['Is Current']][['Version', 'Avg Rating', 'Pct Negative', 'Is Worst', 'Is Current']]

## 5. tab_04 · VoC Feed

Curated sample of representative reviews — 5 per category from negative reviews.

In [ ]:
feed = (
    df[df['is_negative']]
    .groupby('category', group_keys=False)
    .apply(lambda x: x.nlargest(5, 'thumbsUpCount'))
    .reset_index(drop=True)
)

tab_04 = feed[['score', 'content', 'category', 'thumbsUpCount', 'reviewCreatedVersion', 'at']].copy()
tab_04.columns = ['Rating', 'Review Text', 'Category', 'Thumbs Up', 'Version', 'Date']

tab_04.to_csv('tab_04_voc_feed.csv', index=False)
print(f"tab_04: {len(tab_04)} reviews across {tab_04['Category'].nunique()} categories")
tab_04.head(3)